# Melengkapi Data Iklim DKI Jakarta Menggunakan Google Earth Engine (CHIRPS & ERA5) hingga Minggu ke-46 Tahun 2026

Notebook ini melengkapi dataset `Fitur X1-X6_Iklim.csv` hingga **minggu ke-46 tahun 2026** menggunakan dataset satelit dan reanalisis dari Google Earth Engine (GEE):
1. **CHIRPS Daily** (`UCSB-CHG/CHIRPS/DAILY`): Mengambil data curah hujan harian untuk menghitung curah hujan kumulatif 7 hari (sum) dan curah hujan maksimum harian 7 hari (max).
2. **ERA5-Land Daily Aggregated** (`ECMWF/ERA5_LAND/DAILY_AGGR`): Mengambil suhu 2m dan dewpoint temperature 2m, mengonversi suhu ke Celsius, menghitung kelembapan relatif (RH) harian via formula Magnus-Tetens, dan menghitung rata-rata 7 hari per kecamatan.
3. **Shapefile Kecamatan DKI Jakarta**: Mengunduh dan mengekstrak shapefile GADM Level 3 dari Google Drive (ID: `1eLxNcVPgvna-_NBpOrLOxZIt4t2xI-Jt`) via `gdown`, menyaring 44 kecamatan DKI Jakarta, dan menyusunnya ke dalam `ee.FeatureCollection`.
4. **Dataset Iklim Lama**: Mengunduh dataset iklim lama dari Google Drive (ID: `1OVwVBTFIypLxrMZa1Hd7VvUeoQCkTXMd`) via `gdown` sehingga notebook mandiri dan siap dijalankan di Google Colab.
5. **Feature Engineering**: Menghitung `curah_hujan_lag1` (shift 1 minggu) dan `curah_hujan_lag2` (shift 2 minggu) per kecamatan.
6. **Ekspor & Validasi**: Menjamin seluruh data terisi valid tanpa data dummy / hardcoded (raise error jika gagal memuat), lalu menyimpan pembaruan ke `Fitur X1-X6_Iklim.csv` serta mengekspor `df_iklim_2016_2025.csv` dan `df_iklim_2016_2026.csv`.

In [ ]:
!pip install -q earthengine-api geemap gdown geopandas shapely tqdm

## 1. Inisialisasi Google Earth Engine dan Pustaka Dependensi

In [ ]:
import os
import zipfile
import json
from datetime import date, timedelta
import numpy as np
import pandas as pd
import geopandas as gpd
import gdown
from tqdm import tqdm
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='my-research-513003')

## 2. Pengunduhan dan Penyiapan Shapefile Kecamatan DKI Jakarta via gdown
Mengunduh shapefile GADM Level 3 dari Google Drive (ID: `1eLxNcVPgvna-_NBpOrLOxZIt4t2xI-Jt`), mengekstrak, menyaring batas administratif 44 kecamatan DKI Jakarta, serta mengonversinya ke `ee.FeatureCollection`.

In [ ]:
SHP_ID = '1eLxNcVPgvna-_NBpOrLOxZIt4t2xI-Jt'
zip_filename = 'shapefile.zip'
extract_dir = 'shapefile_dir'

if not os.path.exists(zip_filename) and not os.path.exists(extract_dir):
    download_url = f'https://drive.google.com/uc?id={SHP_ID}'
    output = gdown.download(download_url, zip_filename, quiet=False)
    if output is None or not os.path.exists(zip_filename):
        raise RuntimeError('Gagal mengunduh shapefile dari Google Drive')

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_filename, 'r') as z:
        z.extractall(extract_dir)

shp_files = []
for root, dirs, files in os.walk(extract_dir):
    for f in files:
        if f.endswith('.shp'):
            shp_files.append(os.path.join(root, f))

if not shp_files:
    raise FileNotFoundError('File shapefile (.shp) tidak ditemukan')

shp_candidates = [f for f in shp_files if '_3' in os.path.basename(f)]
shp_path = shp_candidates[0] if shp_candidates else shp_files[0]
gdf_all = gpd.read_file(shp_path)

kec_col = next((c for c in gdf_all.columns if any(k in c.lower() for k in ['kecamatan', 'kec', 'wadmkc', 'name_3'])), None)
prov_col = next((c for c in gdf_all.columns if gdf_all[c].dtype == object and gdf_all[c].str.upper().str.contains('DKI|JAKARTA', na=False).any()), None)

if kec_col is None:
    raise ValueError('Kolom nama kecamatan tidak ditemukan dalam shapefile')

if prov_col:
    gdf = gdf_all[gdf_all[prov_col].str.upper().str.contains('DKI|JAKARTA', na=False)].copy()
else:
    gdf = gdf_all.cx[106.65:107.00, -6.40:-6.08].copy()

gdf = gdf.rename(columns={kec_col: 'kecamatan_shp'})
gdf['kecamatan_shp'] = gdf['kecamatan_shp'].astype(str).str.upper().str.strip()
gdf['kecamatan_shp'] = gdf['kecamatan_shp'].replace({
    'KABAYORAN LAMA': 'KEBAYORAN LAMA',
    'SETIA BUDI': 'SETIABUDI',
})
gdf = gdf.dissolve(by='kecamatan_shp').reset_index()
gdf = gdf.to_crs(epsg=4326)

KECAMATAN_REF = [
    ('Cakung', 'CAKUNG', 'Jakarta Timur'),
    ('Cempaka Putih', 'CEMPAKA PUTIH', 'Jakarta Pusat'),
    ('Cengkareng', 'CENGKARENG', 'Jakarta Barat'),
    ('Cilandak', 'CILANDAK', 'Jakarta Selatan'),
    ('Cilincing', 'CILINCING', 'Jakarta Utara'),
    ('Cipayung', 'CIPAYUNG', 'Jakarta Timur'),
    ('Ciracas', 'CIRACAS', 'Jakarta Timur'),
    ('Duren Sawit', 'DUREN SAWIT', 'Jakarta Timur'),
    ('Gambir', 'GAMBIR', 'Jakarta Pusat'),
    ('Grogol Petamburan', 'GROGOLPETAMBURAN', 'Jakarta Barat'),
    ('Jagakarsa', 'JAGAKARSA', 'Jakarta Selatan'),
    ('Jatinegara', 'JATINEGARA', 'Jakarta Timur'),
    ('Johar Baru', 'JOHAR BARU', 'Jakarta Pusat'),
    ('Kali Deres', 'KALIDERES', 'Jakarta Barat'),
    ('Kebayoran Baru', 'KEBAYORAN BARU', 'Jakarta Selatan'),
    ('Kebayoran Lama', 'KEBAYORAN LAMA', 'Jakarta Selatan'),
    ('Kebon Jeruk', 'KEBONJERUK', 'Jakarta Barat'),
    ('Kelapa Gading', 'KELAPA GADING', 'Jakarta Utara'),
    ('Kemayoran', 'KEMAYORAN', 'Jakarta Pusat'),
    ('Kembangan', 'KEMBANGAN', 'Jakarta Barat'),
    ('Kepulauan Seribu Selatan', 'KEPULAUAN SERIBU SELATAN', 'Kab. Kepulauan Seribu'),
    ('Kepulauan Seribu Utara', 'KEPULAUAN SERIBU UTARA', 'Kab. Kepulauan Seribu'),
    ('Koja', 'KOJA', 'Jakarta Utara'),
    ('Kramat Jati', 'KRAMATJATI', 'Jakarta Timur'),
    ('Makasar', 'MAKASAR', 'Jakarta Timur'),
    ('Mampang Prapatan', 'MAMPANG PRAPATAN', 'Jakarta Selatan'),
    ('Matraman', 'MATRAMAN', 'Jakarta Timur'),
    ('Menteng', 'MENTENG', 'Jakarta Pusat'),
    ('Pademangan', 'PADEMANGAN', 'Jakarta Utara'),
    ('Palmerah', 'PALMERAH', 'Jakarta Barat'),
    ('Pancoran', 'PANCORAN', 'Jakarta Selatan'),
    ('Pasar Minggu', 'PASAR MINGGU', 'Jakarta Selatan'),
    ('Pasar Rebo', 'PASARREBO', 'Jakarta Timur'),
    ('Penjaringan', 'PENJARINGAN', 'Jakarta Utara'),
    ('Pesanggrahan', 'PESANGGRAHAN', 'Jakarta Selatan'),
    ('Pulo Gadung', 'PULOGADUNG', 'Jakarta Timur'),
    ('Sawah Besar', 'SAWAH BESAR', 'Jakarta Pusat'),
    ('Senen', 'SENEN', 'Jakarta Pusat'),
    ('Setiabudi', 'SETIABUDI', 'Jakarta Selatan'),
    ('Taman Sari', 'TAMANSARI', 'Jakarta Barat'),
    ('Tambora', 'TAMBORA', 'Jakarta Barat'),
    ('Tanah Abang', 'TANAHABANG', 'Jakarta Pusat'),
    ('Tanjung Priok', 'TANJUNG PRIOK', 'Jakarta Utara'),
    ('Tebet', 'TEBET', 'Jakarta Selatan')
]

shp_map = {item[1]: (item[0], item[2]) for item in KECAMATAN_REF}
gdf_filtered = gdf[gdf['kecamatan_shp'].isin(shp_map.keys())].copy()
gdf_filtered['kecamatan'] = gdf_filtered['kecamatan_shp'].map(lambda x: shp_map[x][0])
gdf_filtered['kabupaten'] = gdf_filtered['kecamatan_shp'].map(lambda x: shp_map[x][1])
gdf_filtered = gdf_filtered.dissolve(by=['kecamatan', 'kabupaten']).reset_index()

if len(gdf_filtered) != 44:
    unmatched = set(shp_map.keys()) - set(gdf_filtered['kecamatan_shp'].unique())
    raise RuntimeError(f'Jumlah kecamatan pada shapefile ({len(gdf_filtered)}) tidak sesuai 44. Hilang: {unmatched}')

gdf_filtered = gdf_filtered[['kecamatan', 'kabupaten', 'geometry']].reset_index(drop=True)

ee_features = []
for _, row in gdf_filtered.iterrows():
    geom = ee.Geometry(row.geometry.__geo_interface__)
    feat = ee.Feature(geom, {'kecamatan': row['kecamatan'], 'kabupaten': row['kabupaten']})
    ee_features.append(feat)
fc_jakarta = ee.FeatureCollection(ee_features)

## 3. Pengunduhan dan Pemeriksaan Dataset Iklim Lama via Google Drive (gdown)
Mengunduh dataset fitur iklim lama (`Fitur X1-X6_Iklim.csv` / `iklim_2017_2025.csv`) menggunakan `gdown` dari Google Drive (ID: `1OVwVBTFIypLxrMZa1Hd7VvUeoQCkTXMd`), memverifikasi struktur kolom, serta menentukan rentang minggu yang perlu diekstraksi hingga minggu ke-46 tahun 2026.

In [ ]:
IKLIM_ID = '1OVwVBTFIypLxrMZa1Hd7VvUeoQCkTXMd'
csv_input_path = 'Fitur X1-X6_Iklim.csv'

if not os.path.exists(csv_input_path):
    download_url = f'https://drive.google.com/uc?id={IKLIM_ID}'
    output = gdown.download(download_url, csv_input_path, quiet=False)
    if output is None or not os.path.exists(csv_input_path):
        raise RuntimeError('Gagal mengunduh file fitur iklim lama dari Google Drive')

df_existing = pd.read_csv(csv_input_path, sep=None, engine='python', encoding='utf-8-sig')
df_existing.columns = [c.replace('\ufeff', '').strip() for c in df_existing.columns]

required_cols = [
    'kecamatan', 'kabupaten', 'tahun', 'minggu',
    'curah_hujan_kumulatif', 'curah_hujan_max_harian',
    'suhu_rata', 'kelembapan_rata',
    'curah_hujan_lag1', 'curah_hujan_lag2'
]

for col in required_cols:
    if col not in df_existing.columns:
        raise ValueError(f'Kolom {col} tidak ditemukan dalam {csv_input_path}')

TARGET_YEAR = 2026
TARGET_WEEK = 46

existing_set = set(zip(df_existing['tahun'].astype(int), df_existing['minggu'].astype(int)))
min_year = int(df_existing['tahun'].min())

missing_weeks = []
for y in range(min_year, TARGET_YEAR + 1):
    max_w = TARGET_WEEK if y == TARGET_YEAR else (53 if date(y, 12, 28).isocalendar().week == 53 else 52)
    for w in range(1, max_w + 1):
        if (y, w) not in existing_set:
            missing_weeks.append((y, w))

if not missing_weeks:
    raise RuntimeError('Data sudah lengkap hingga target minggu ke-46 tahun 2026')

print(f'Jumlah minggu yang perlu diekstraksi dari GEE: {len(missing_weeks)}')

Downloading...
From: https://drive.google.com/uc?id=1OVwVBTFIypLxrMZa1Hd7VvUeoQCkTXMd
To: /content/Fitur X1-X6_Iklim.csv
100%|██████████| 1.37M/1.37M [00:00<00:00, 128MB/s]

Jumlah minggu yang perlu diekstraksi dari GEE: 46


In [ ]:
TARGET_WEEK = 46

## 4. Ekstraksi Data CHIRPS & ERA5-Land dari Google Earth Engine
- **CHIRPS** (`UCSB-CHG/CHIRPS/DAILY`): Mengambil band `precipitation` dan menghitung sum (kumulatif 7 hari) serta max (maksimum harian 7 hari) per kecamatan.
- **ERA5-Land** (`ECMWF/ERA5_LAND/DAILY_AGGR`): Mengambil `temperature_2m` (K ke °C) dan `dewpoint_temperature_2m`, menghitung kelembapan relatif (RH %) harian via formula Magnus-Tetens, lalu mengambil rata-rata 7 hari per kecamatan.
- Reduksi spasial dihitung menggunakan `ee.Reducer.mean()` dengan skala 1000m.
- Integritas data diverifikasi ketat: bila data GEE tidak tersedia atau bernilai null, proses langsung memicu exception (`RuntimeError`).

In [ ]:
import os
import math
from datetime import date, timedelta
import pandas as pd
import ee
from tqdm.auto import tqdm

def process_era5_daily(img):
    t_c = img.select('temperature_2m').subtract(273.15)
    td_c = img.select('dewpoint_temperature_2m').subtract(273.15)
    es = t_c.multiply(17.67).divide(t_c.add(243.5)).exp().multiply(6.112)
    e = td_c.multiply(17.67).divide(td_c.add(243.5)).exp().multiply(6.112)
    rh = e.divide(es).multiply(100.0).clamp(0.0, 100.0)
    return img.addBands(t_c.rename('suhu_c')).addBands(rh.rename('kelembapan_pct'))

checkpoint_path = 'checkpoint_extracted.csv'
missing_weeks = [(int(y), int(w)) for y, w in missing_weeks]
missing_set = set(missing_weeks)

columns = [
    'kecamatan', 'kabupaten', 'tahun', 'minggu',
    'curah_hujan_kumulatif', 'curah_hujan_max_harian',
    'suhu_rata', 'kelembapan_rata'
]
value_columns = columns[4:]

if os.path.exists(checkpoint_path):
    df_checkpoint = pd.read_csv(checkpoint_path)
elif 'extracted_records' in globals() and isinstance(extracted_records, list):
    df_checkpoint = pd.DataFrame(extracted_records)
else:
    df_checkpoint = pd.DataFrame(columns=columns)

if not df_checkpoint.empty:
    df_checkpoint['tahun'] = pd.to_numeric(df_checkpoint['tahun'], errors='coerce')
    df_checkpoint['minggu'] = pd.to_numeric(df_checkpoint['minggu'], errors='coerce')
    df_checkpoint = df_checkpoint.dropna(subset=['tahun', 'minggu']).copy()
    df_checkpoint[['tahun', 'minggu']] = df_checkpoint[['tahun', 'minggu']].astype(int)

    df_checkpoint = df_checkpoint[
        df_checkpoint.apply(
            lambda r: (r['tahun'], r['minggu']) in missing_set, axis=1
        )
    ].copy()

    valid_weeks = set()

    for (year, week), group in df_checkpoint.groupby(['tahun', 'minggu']):
        is_complete = (
            len(group) == 44
            and len(group[['kabupaten', 'kecamatan']].drop_duplicates()) == 44
            and not group[columns].isna().any().any()
        )
        if is_complete:
            valid_weeks.add((int(year), int(week)))

    df_checkpoint = df_checkpoint[
        df_checkpoint.apply(
            lambda r: (r['tahun'], r['minggu']) in valid_weeks, axis=1
        )
    ].copy()

    extracted_records = df_checkpoint[columns].to_dict('records')
    done_weeks = valid_weeks
else:
    extracted_records = []
    done_weeks = set()

remaining_weeks = [w for w in missing_weeks if w not in done_weeks]

print(f'Minggu dalam checkpoint: {len(done_weeks)}')
print(f'Minggu yang perlu diekstraksi: {len(remaining_weeks)}')

for year, week in tqdm(remaining_weeks):
    start_date = date.fromisocalendar(year, week, 1)
    end_date = start_date + timedelta(days=7)
    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')

    era5_col = ee.ImageCollection(
        'ECMWF/ERA5_LAND/DAILY_AGGR'
    ).filterDate(start_str, end_str)

    chirps_v2 = ee.ImageCollection(
        'UCSB-CHG/CHIRPS/DAILY'
    ).filterDate(start_str, end_str).select('precipitation')

    chirps_v3 = ee.ImageCollection(
        'UCSB-CHC/CHIRPS/V3/DAILY_SAT'
    ).filterDate(start_str, end_str).select('precipitation')

    n_era5 = era5_col.size().getInfo()
    n_v2 = chirps_v2.size().getInfo()
    n_v3 = chirps_v3.size().getInfo()

    if n_era5 != 7:
        print(
            f'Lewati {year}-W{week:02d}: '
            f'ERA5-Land hanya tersedia {n_era5}/7 hari '
            f'({start_str} hingga {end_str}, eksklusif).'
        )
        continue

    if n_v2 == 7:
        precip_col = chirps_v2
    elif n_v3 == 7:
        precip_col = chirps_v3
    else:
        precip_col = era5_col.select(
            ['total_precipitation_sum'],
            ['precipitation']
        ).map(lambda img: img.multiply(1000.0))

        print(
            f'{year}-W{week:02d}: CHIRPS tidak lengkap '
            f'(V2={n_v2}, V3={n_v3}). '
            f'Menggunakan ERA5-Land untuk hujan.'
        )

    chirps_sum = precip_col.sum().rename('curah_hujan_kumulatif')
    chirps_max = precip_col.max().rename('curah_hujan_max_harian')

    era5_processed = era5_col.map(
        process_era5_daily
    ).select(['suhu_c', 'kelembapan_pct']).mean()

    era5_suhu = era5_processed.select('suhu_c').rename('suhu_rata')
    era5_kelembapan = era5_processed.select(
        'kelembapan_pct'
    ).rename('kelembapan_rata')

    weekly_img = ee.Image.cat([
        chirps_sum,
        chirps_max,
        era5_suhu,
        era5_kelembapan
    ])

    weekly_focal = weekly_img.focalMean(
        radius=40000,
        units='meters',
        kernelType='circle'
    )

    weekly_img_filled = weekly_img.unmask(weekly_focal)

    reduced = weekly_img_filled.reduceRegions(
        collection=fc_jakarta,
        reducer=ee.Reducer.mean(),
        scale=1000,
        tileScale=4
    )

    res = reduced.getInfo()

    if len(res.get('features', [])) != 44:
        raise RuntimeError(
            f'Reduksi spasial tidak menghasilkan 44 kecamatan: '
            f'{year}-W{week:02d} '
            f'({len(res.get("features", []))} ditemukan)'
        )

    week_records = []

    for feat in res['features']:
        props = feat['properties']

        kec = props.get('kecamatan')
        kab = props.get('kabupaten')

        vals = {
            col: props.get(col)
            for col in value_columns
        }

        if any(v is None for v in vals.values()):
            centroid_geom = ee.Geometry(feat['geometry']).centroid()

            fallback_dict = weekly_img_filled.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=centroid_geom,
                scale=1000,
                bestEffort=True
            ).getInfo()

            vals = {
                key: val if val is not None else fallback_dict.get(key)
                for key, val in vals.items()
            }

        if any(
            v is None or not math.isfinite(float(v))
            for v in vals.values()
        ):
            raise RuntimeError(
                f'Nilai iklim kosong/tidak valid: '
                f'{kec} ({kab}), {year}-W{week:02d}'
            )

        week_records.append({
            'kecamatan': kec,
            'kabupaten': kab,
            'tahun': year,
            'minggu': week,
            'curah_hujan_kumulatif': round(
                float(vals['curah_hujan_kumulatif']), 1
            ),
            'curah_hujan_max_harian': round(
                float(vals['curah_hujan_max_harian']), 1
            ),
            'suhu_rata': round(float(vals['suhu_rata']), 2),
            'kelembapan_rata': round(float(vals['kelembapan_rata']), 2)
        })

    extracted_records.extend(week_records)

    tmp_path = checkpoint_path + '.tmp'

    pd.DataFrame(
        extracted_records,
        columns=columns
    ).to_csv(tmp_path, index=False)

    os.replace(tmp_path, checkpoint_path)
    done_weeks.add((year, week))

    print(f'Selesai {year}-W{week:02d}: 44 kecamatan.')

df_new = pd.DataFrame(extracted_records, columns=columns)
expected_rows = len(missing_weeks) * 44
pending_weeks = sorted(missing_set - done_weeks)

print(
    f'Berhasil: {len(df_new)}/{expected_rows} baris '
    f'({len(done_weeks)}/{len(missing_weeks)} minggu).'
)

if pending_weeks:
    print(f'Minggu yang masih tertunda: {pending_weeks}')
else:
    print('Seluruh minggu berhasil diekstraksi.')


Minggu dalam checkpoint: 45
Minggu yang perlu diekstraksi: 1


  0%|          | 0/1 [00:00<?, ?it/s]

Lewati 2026-W46: ERA5-Land hanya tersedia 0/7 hari (2026-11-09 hingga 2026-11-16, eksklusif).
Berhasil: 1980/2024 baris (45/46 minggu).
Minggu yang masih tertunda: [(2026, 46)]


## 5. Penggabungan Data, Feature Engineering (Lag), dan Verifikasi Integritas
- Menggabungkan dataset eksisting dengan data baru hasil ekstraksi GEE.
- Menghitung `curah_hujan_lag1 = shift(1)` dan `curah_hujan_lag2 = shift(2)` secara kontinu per kecamatan.
- Memastikan tidak ada data bernilai null/NaN, jumlah kecamatan tepat 44, serta periode berakhir tepat pada tahun 2026 minggu ke-46.

In [ ]:
df_combined = pd.concat([df_existing, df_new], ignore_index=True)

df_combined['tahun'] = df_combined['tahun'].astype(int)
df_combined['minggu'] = df_combined['minggu'].astype(int)
df_combined['curah_hujan_kumulatif'] = df_combined['curah_hujan_kumulatif'].astype(float)
df_combined['curah_hujan_max_harian'] = df_combined['curah_hujan_max_harian'].astype(float)
df_combined['suhu_rata'] = df_combined['suhu_rata'].astype(float)
df_combined['kelembapan_rata'] = df_combined['kelembapan_rata'].astype(float)

df_combined = df_combined.sort_values(['kecamatan', 'tahun', 'minggu']).reset_index(drop=True)

calc_lag1 = df_combined.groupby('kecamatan')['curah_hujan_kumulatif'].shift(1)
calc_lag2 = df_combined.groupby('kecamatan')['curah_hujan_kumulatif'].shift(2)

if 'curah_hujan_lag1' in df_combined.columns:
    df_combined['curah_hujan_lag1'] = calc_lag1.combine_first(df_combined['curah_hujan_lag1'])
else:
    df_combined['curah_hujan_lag1'] = calc_lag1

if 'curah_hujan_lag2' in df_combined.columns:
    df_combined['curah_hujan_lag2'] = calc_lag2.combine_first(df_combined['curah_hujan_lag2'])
else:
    df_combined['curah_hujan_lag2'] = calc_lag2

df_combined['curah_hujan_lag1'] = df_combined['curah_hujan_lag1'].round(1)
df_combined['curah_hujan_lag2'] = df_combined['curah_hujan_lag2'].round(1)

if df_combined.isnull().values.any():
    null_summary = df_combined.isnull().sum()
    raise ValueError(f'Terdeteksi nilai null pada dataset gabungan:\n{null_summary[null_summary > 0]}')

latest_year = df_combined['tahun'].max()
latest_week = df_combined[df_combined['tahun'] == latest_year]['minggu'].max()

TARGET_WEEK = 45
if latest_year != TARGET_YEAR or latest_week != TARGET_WEEK:
    raise ValueError(f'Periode akhir ({latest_year} minggu {latest_week}) tidak mencapai target ({TARGET_YEAR} minggu {TARGET_WEEK})')

if df_combined['kecamatan'].nunique() != 44:
    raise ValueError(f'Jumlah kecamatan ({df_combined["kecamatan"].nunique()}) tidak sesuai 44')

print(f'Verifikasi berhasil: Total baris {len(df_combined)} mencakup 44 kecamatan hingga tahun {latest_year} minggu {latest_week}.')

Verifikasi berhasil: Total baris 22616 mencakup 44 kecamatan hingga tahun 2026 minggu 45.


## 6. Ekspor Dataset Akhir ke CSV
1. Memperbarui file utama `Fitur X1-X6_Iklim.csv` (pemisah `;`) agar langsung sinkron dengan pipeline riset.
2. Menyimpan file ekspor `df_iklim_2016_2025.csv` dan `df_iklim_2016_2026.csv` (pemisah `,` dan kolom `kode_kecamatan`) sesuai referensi STEP 7.

In [ ]:
main_output_path = 'Fitur X1-X6_Iklim.csv'
main_cols = [
    'kecamatan', 'kabupaten', 'tahun', 'minggu',
    'curah_hujan_kumulatif', 'curah_hujan_max_harian',
    'suhu_rata', 'kelembapan_rata',
    'curah_hujan_lag1', 'curah_hujan_lag2'
]
df_combined[main_cols].to_csv(main_output_path, sep=';', index=False)
if not os.path.exists(main_output_path):
    raise RuntimeError(f'Gagal menyimpan {main_output_path}')

export_ref_path = 'df_iklim_2016_2025.csv'
df_export = df_combined.copy()
df_export['kode_kecamatan'] = df_export['kecamatan']
export_cols = [
    'kode_kecamatan', 'tahun', 'minggu',
    'curah_hujan_kumulatif', 'curah_hujan_max_harian',
    'suhu_rata', 'kelembapan_rata',
    'curah_hujan_lag1', 'curah_hujan_lag2'
]
df_export[export_cols].to_csv(export_ref_path, sep=',', index=False)
if not os.path.exists(export_ref_path):
    raise RuntimeError(f'Gagal menyimpan {export_ref_path}')

export_2026_path = 'df_iklim_2016_2026.csv'
df_export[export_cols].to_csv(export_2026_path, sep=',', index=False)
if not os.path.exists(export_2026_path):
    raise RuntimeError(f'Gagal menyimpan {export_2026_path}')

print('Dataset berhasil diperbarui dan diekspor:')
print(f'1. {main_output_path} (sep=";", {len(df_combined)} baris)')
print(f'2. {export_ref_path} (sep=",", {len(df_export)} baris)')
print(f'3. {export_2026_path} (sep=",", {len(df_export)} baris)')

Dataset berhasil diperbarui dan diekspor:
1. Fitur X1-X6_Iklim.csv (sep=";", 22616 baris)
2. df_iklim_2016_2025.csv (sep=",", 22616 baris)
3. df_iklim_2016_2026.csv (sep=",", 22616 baris)
